# Exploración de `inscripciones.csv`
Tema 07 - Instituto Superior Horizonte

Antes de escribir las reglas de validación miro cómo vienen los datos. Todo se lee como texto para no perder información.

In [1]:
import pandas as pd

df = pd.read_csv('datos/inscripciones.csv', dtype=str, keep_default_na=False)
print(df.shape)
df.head()

(2575, 9)


,id_inscripcion,id_alumno,nombre_alumno,carrera,materia,fecha_inscripcion,nota_parcial,nota_final,condicion
0,I00001,274,Agustina Ruiz,Analista de Datos,Estadistica,2024-12-25,6.8,8.6,aprobado
1,I00002,88,Lucia Vega,Tecnicatura en Redes,Ingles Tecnico,2025-10-07,4.2,4.0,desaprobado
2,I00003,400,Juan Ramos,Tecnicatura en Redes,Matematica,2025-07-10,42,2.6,desaprobado
3,I00004,253,Tomas Medina,Analista de Datos,Base de Datos,2025-08-01,92,4.3,desaprobado
4,I00005,380,Mateo Sosa,Analista de Datos,Estadistica,2025-01-05,4.3,8.6,aprobado


## Valores vacíos y duplicados

In [2]:
print('vacíos por columna:')
print((df == '').sum())
print()
print('id_inscripcion repetidos:', df['id_inscripcion'].duplicated().sum())
print('filas completamente repetidas:', df.duplicated().sum())

vacíos por columna:
id_inscripcion         0
id_alumno              0
nombre_alumno          0
carrera                0
materia                0
fecha_inscripcion      0
nota_parcial           0
nota_final             0
condicion            155
dtype: int64

id_inscripcion repetidos: 75
filas completamente repetidas: 75


## Columnas categóricas (carrera, materia, condicion)

In [3]:
for col in ['carrera', 'materia', 'condicion']:
    print(df[col].value_counts(dropna=False))
    print()

carrera
Analista de Datos          873
Tecnicatura en Software    872
Tecnicatura en Redes       830
Name: count, dtype: int64

materia
Matematica             617
Ingles Tecnico         436
Base de Datos          425
Estadistica            235
Sistemas Operativos    225
Python para Datos      215
Redes 1                214
Programacion 1         208
Name: count, dtype: int64

condicion
desaprobado     1154
aprobado        1138
                 155
APROBADO          27
 APROBADO         26
 DESAPROBADO      21
DESAPROBADO       20
APROBADO          18
DESAPROBADO       16
Name: count, dtype: int64



`condicion` viene con vacíos, mayúsculas y espacios: `'APROBADO'`, `' DESAPROBADO'`, etc.

## Fechas

In [4]:
iso = df['fecha_inscripcion'].str.fullmatch(r'\d{4}-\d{2}-\d{2}')
print('formato YYYY-MM-DD:', iso.sum())
print('otros formatos:', (~iso).sum())
df.loc[~iso, 'fecha_inscripcion'].head()

formato YYYY-MM-DD: 2318
otros formatos: 257


12    09/06/2024
15    02/01/2025
24    05/07/2025
32    30/10/2024
38    17/09/2024
Name: fecha_inscripcion, dtype: str

In [5]:
# ¿se pueden pasar todas a fecha real?
f1 = pd.to_datetime(df['fecha_inscripcion'], format='%Y-%m-%d', errors='coerce')
f2 = pd.to_datetime(df['fecha_inscripcion'], format='%d/%m/%Y', errors='coerce')
print('sin poder convertir:', (f1.isna() & f2.isna()).sum())
f = f1.fillna(f2)
print(f.min(), f.max())

sin poder convertir: 0
2024-01-01 00:00:00 2025-12-30 00:00:00


## Notas (deberían ir de 1 a 10)

In [6]:
for col in ['nota_parcial', 'nota_final']:
    n = pd.to_numeric(df[col], errors='coerce')
    print(col, '| no numéricas:', n.isna().sum(), '| mínimo:', n.min(), '| máximo:', n.max())
    print('   fuera de 1-10:', ((n < 1) | (n > 10)).sum(), '| valores fuera de rango entre 11 y 19:', ((n > 10) & (n < 20)).sum())
    print('   ejemplos fuera de rango:', sorted(df.loc[n > 10, col].unique())[:12])

nota_parcial | no numéricas: 0 | mínimo: 2.0 | máximo: 99.0
   fuera de 1-10: 131 | valores fuera de rango entre 11 y 19: 0
   ejemplos fuera de rango: ['20', '21', '22', '24', '25', '27', '28', '29', '31', '32', '33', '34']
nota_final | no numéricas: 0 | mínimo: 2.0 | máximo: 99.0
   fuera de 1-10: 154 | valores fuera de rango entre 11 y 19: 0
   ejemplos fuera de rango: ['20', '22', '23', '24', '25', '26', '27', '30', '31', '32', '33', '34']


Las notas fuera de rango son números de dos cifras sin punto (42, 68, 92...): parecen notas como 4.2, 6.8, 9.2 con el punto decimal perdido. Para comprobarlo veo si, al dividir por 10, la `condicion` es coherente con `nota_final >= 6`.

In [7]:
nf = pd.to_numeric(df['nota_final'])
nf_corr = nf.where(nf <= 10, nf / 10)
cond = df['condicion'].str.strip().str.lower()
esperada = (nf_corr >= 6).map({True: 'aprobado', False: 'desaprobado'})
con_dato = cond != ''
print('registros con condicion cargada:', con_dato.sum())
print('contradicen la nota (dividiendo por 10):', (con_dato & (cond != esperada)).sum())

registros con condicion cargada: 2420
contradicen la nota (dividiendo por 10): 0


Da 0 contradicciones, así que la hipótesis es correcta: se puede corregir dividiendo por 10.

## Nombres

In [8]:
n = df['nombre_alumno']
print('con espacios al principio o al final:', (n != n.str.strip()).sum())
print('todo en mayúscula:', (n == n.str.upper()).sum())
print('palabras por nombre:', n.str.strip().str.split().str.len().unique())

con espacios al principio o al final: 138
todo en mayúscula: 207
palabras por nombre: [2]


## Resumen de problemas encontrados
- 75 filas duplicadas (mismo `id_inscripcion`, registro idéntico).
- 257 fechas en formato `dd/mm/yyyy`.
- 285 notas con el punto decimal perdido (131 en `nota_parcial`, 154 en `nota_final`).
- `condicion` vacía (155) o con mayúsculas/espacios.
- Nombres con espacios sobrantes y/o todo en mayúscula (207 registros).
- `carrera`, `materia`, `id_alumno` e `id_inscripcion` no tienen problemas de formato.